#Prerequisites


## Create Catalog, Schema and Delta table

In [0]:

%sql
-- CREATE CATALOG IF NOT EXISTS quickstart_catalog;
USE CATALOG quickstart_catalog;
-- CREATE SCHEMA IF NOT EXISTS quickstart_schema;
CREATE TABLE quickstart_schema.users (
    id INT,
    name STRING,
    dob DATE,
    email STRING,
    gender STRING,
    country STRING,
    region STRING,
    city STRING,
    asset INT,
    marital_status STRING
  );
 
DESCRIBE EXTENDED quickstart_schema.users;

#Transaction 02- load data into Datatable

In [0]:
df = spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
).write.mode("overwrite").saveAsTable("quickstart_schema.users")

#Transaction 03- Filter country = "India

In [0]:
from pyspark.sql.functions import col
df = spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
).filter(col("country")=="India").write.mode("overwrite").saveAsTable("quickstart_schema.users")

#Transaction 04- Filter country ="United States"

In [0]:
from pyspark.sql.functions import col
df = spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
).filter(col("country")=="United States").write.mode("overwrite").saveAsTable("quickstart_schema.users")

#Transaction 05- Filter country = Finland
- even for the Finland data is not present then also transaction is created just it will show 0 items

In [0]:
from pyspark.sql.functions import col
df = spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
).filter(col("country")=="Finland").write.mode("overwrite").saveAsTable("quickstart_schema.users")

#Versioning

##using pyspark- Read Delta Table (by Default Latest Version)

In [0]:
spark.read.option("versionAsOf",1).table("quickstart_schema.users").display()

## Using sql approach

In [0]:
%sql

SELECT * from quickstart_schema.users VERSION AS OF 1;

#List Transaction History

## Using DeltaTable API

In [0]:
from delta.tables import DeltaTable

delta_table= DeltaTable.forName(spark, "quickstart_schema.users")

delta_table.history().display()


##using SQL

In [0]:
%sql
DESCRIBE HISTORY quickstart_schema.users;


# Time travel

##using Pyspark 

In [0]:
spark.read.option("timestampAsOf", "2026-05-11T06:11:49.000+00:00").table("quickstart_schema.users").limit(4).display()

##Using SQL

In [0]:
%sql

SELECT * from quickstart_schema.users TIMESTAMP AS OF "2026-05-11T06:11:49.000+00:00" LIMIT 4;

#Restore

##using SQL

In [0]:
%sql 
RESTORE TABLE quickstart_schema.users TO VERSION AS OF 1;

In [0]:
%sql

SELECT * from quickstart_schema.users VERSION AS OF 1;

#Transaction 06 - Add transaction message using userMetaData

##transaction Message Simple

In [0]:
from pyspark.sql.functions import col

df = (
    spark.read.csv(
        path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
        header=True,
        inferSchema=True,
    )
    .filter(col("region") == "Karnataka")
    .write.option("userMetadata", "Loaded data for region='Karnataka'")
    .mode("overwrite")
    .saveAsTable("quickstart_schema.users")
)

##Transaction Message Complex

In [0]:
import json
from datetime import datetime
 
transaction_metadata = {
    "pipeline": "users_daily_ingestion",
    "job_name": "DAILY_USERS_JOB",
    "trigger": "SCHEDULED",
    "load_time": datetime.now().isoformat(),
}
 
 
spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
).filter(col("region") == "Bihar").write.option(
    "userMetadata", json.dumps(transaction_metadata)
).mode(
    "overwrite"
).saveAsTable(
    "quickstart_schema.users"
)